In [1]:
import ollama

file_path = 'myd.md'
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()
paragraphs = content.split('##')

for i,paragraph in enumerate(paragraphs):
    print(f'paragraph {i + 1}:\n{paragraph}\n')
    print('-' * 20)

paragraph 1:
---
title: 'FAQ'
category: 'faq'
---

# 常见问题



--------------------
paragraph 2:
 1. openGauss 数据库开源许可协议是什么？

木兰宽松许可证 MulanPSL2 V2，无传染



--------------------
paragraph 3:
 2. openGauss 支持部署的硬件架构及操作系统有哪些？

ARM：openEuler 20.03LTS（推荐采用此操作系统）｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ FusionOS 22 ｜统信

X86：openEuler 20.03LTS ｜ openEuler 22.03LTS ｜ Kylin-V10 ｜ CentOS 7.6 ｜ Asianux 7.6 ｜ FusionOS 22 ｜统信

<br/>

ubuntu / centos8 /centos10 / 红旗 需要适配编译数据库；在飞腾/海光 等服务器上安装需要重新适配编译。



--------------------
paragraph 4:
 3. openGauss 有哪些版本？

openGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：

1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；
2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；
3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；
4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。

详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。



--------------------
paragraph 5:
 4. openGauss 分布式部署方案是什么？

1. 基于 openLookeng 实现分布式分析能力，与 shardingsphere 配合 openGau

In [2]:
def embedding(text):
    vector = ollama.embeddings(model="nomic-embed-text", prompt=text)
    return vector["embedding"]

In [3]:
text = "openGauss 是一款开源数据库"
emb = embedding(text)
dimensions = len(emb)

print("text: {}, embedding dim : {}, embedding : {}...".format(text, dimensions, emb[:10]))


text: openGauss 是一款开源数据库, embedding dim : 768, embedding : [-0.5392146110534668, 1.3381578922271729, -3.5275259017944336, -1.0050768852233887, -0.19613319635391235, 0.2840339243412018, -0.4675346612930298, 0.08503472805023193, -0.22875681519508362, -0.9965018033981323]...


In [4]:
import psycopg2

table_name = "opengauss_data"
conn = psycopg2.connect(
    database = "db1",
    user="user1",
    password="Test@123",
    host="127.0.0.1",
    port="5432"
)

cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS {};".format(table_name))
cur.execute("CREATE TABLE {} (id INT PRIMARY KEY, content TEXT, emb vector({}))".format(table_name,dimensions))
conn.commit()

In [5]:
for i,paragraph in enumerate(paragraphs):
    emb = embedding(paragraph)
    insert_data_sql = f'''INSERT INTO {table_name} (id, content, emb) VALUES (%s, %s, %s);'''
    cur.execute(insert_data_sql, (i, paragraph, emb))
    conn.commit()

cur.execute("CREATE INDEX ON {} USING HNSW (emb vector_l2_ops);".format(table_name))
conn.commit()


In [6]:
question = "openGauss 发布了哪些版本？"

emb_data = embedding(question)
dimensions = len(emb_data)

cur = conn.cursor()
cur.execute("select content from {} order by emb <-> '{}' limit 1;".format(table_name, emb_data))
conn.commit()

rows = cur.fetchall()
print(rows)

cur.close()
conn.close()

[(' 3. openGauss 有哪些版本？\n\nopenGauss 社区每两年发布一个 LTS 版本，LTS 版本作为长期支持版本，可规模上线使用。半年发布一个创新版本，创新版本供用户联创测试使用；涉及重大问题修复时，会按需发布补丁版本。同时按照不同场景分为以下版本：\n\n1. openGauss 企业版:具备更齐全的集群管理功能,适合企业用户；\n2. openGauss 极简版:安装配置简单,解压可用,适合个人开发者；\n3. openGauss 轻量版:精简功能,缩减安装包大小,内存占用更少；\n4. openGauss 分布式镜像:基于 ShardingSphere 和 k8s 的分布式容器化镜像。\n\n详情参考 openGauss 官网[“学习”->“文档”](https://docs.opengauss.org)区域。\n\n',)]


In [7]:
context = "\n".join(row[0] for row in rows)
SYSTEM_PROMPT = "你作为一个对话 AI 助手，结合上下文信息简练高效地回答用户提出的问题"
USER_PROMPT = f"请结合{context}信息来回答{question}的问题，不需要额外的无用回答"


response = ollama.chat(
    model = "deepseek-r1",
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
)
print(response["message"]["content"])

<think>
嗯，用户问的是“openGauss 发布了哪些版本？”，我需要结合提供的上下文信息来回答。首先，看看用户给的信息：

1. openGauss 社区每两年发布一个 LTS 版本。
2. 还有半年发布创新版本供联创测试。
3. 重大问题修复时会发布补丁版本。
4. 按场景分为企业版、极简版、轻量版和分布式镜像。

所以，直接提取这些信息，把各个版本名称列出来就行。不用添加额外内容，保持简洁高效。可能用户是想了解当前可用的版本，或者进行对比选择适合自己的。因此，列出四个主要版本：LTS、创新和补丁，加上详细说明每个版本的目标用户就可以了。
</think>

openGauss 发布了以下版本：

1. LTS 版本 - 作为长期支持版本，可规模上线使用。
2. 创新版本 - 提供联创测试使用。
3. 补丁版本 - 按需发布修复重大问题。

此外，按场景分为：
- 企业版
- 极简版
- 轻量版
- 分布式镜像


In [8]:
USER_PROMPT = f"请回答{question}的问题，不需要额外的无用回答"


response = ollama.chat(
    model = "deepseek-r1",
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
)
print(response["message"]["content"])


<think>
好的，我需要回答用户关于openGauss发布版本的问题。首先，我要确保我的回答准确且简洁。

我知道openGauss目前有多个主要版本，包括OGB-0.1、OGB-0.2和OGB-Lite系列。OGB-0.1是早期版本，OGB-0.2进行了改进，比如增加了外层解析功能，并支持更多数据类型。OGB-Lite则更轻量，适合资源受限的环境。

接下来，我应该按照时间顺序列出这些版本，并简要说明每个版本的主要特点。这样用户可以清楚了解各个版本之间的差异和进化历程。

最后，我要确保不添加额外信息，只提供必要的版本列表和简要描述。
</think>

openGauss 提供了多个版本以适应不同的需求：

1. **OGB-0.1**: 是第一个发布版本，提供了基础的功能。
2. **OGB-0.2**: 增加了外层解析功能，并支持更多的数据类型。
3. **OGB-Lite**: 一个轻量级的版本，适合资源受限的环境。

这些版本逐步扩展了功能和性能，满足了用户在不同场景下的需求。
